In [1]:
import sys
import time
from datetime import datetime
from pathlib import Path
import numpy as np
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from dataset import DTIDataset, MyDataset
from model import DeepDTAF, DeepDTAF_3to2, test, DTIDenseNet3to1, DTIDenseNet3to2, DTIDenseNet2to1, Loss_fc

In [2]:
data_path, fold = '../DTI-dataset', 4
seed = np.random.randint(18885, 18886) # 生成特定范围内的随机数
path = Path(f'/data1/zxy/DTI Results/Dense433_3to1-drop_p_test/3to1_050000_r20_fold{fold}_{datetime.now().strftime("%Y%m%d%H%M%S")}_{seed}')  # 一个存储结果文件的文件夹，并记录随机数种子
path.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda:0")  # or torch.device('cpu')
            
max_seq_len = 904  # 数据预处理使用的值，定义序列长度，蛋白
max_pkt_len = 64  # 口袋
max_smi_len = 154  # 化合物

repeat = 20
fix = False
onehot = False  # smiles是否使用one-hot编码
dilation = True  # 使用扩张卷积
mode = 3   # 输入信息随机失活门控，0:无失活，1:失活口袋，2:失活蛋白，3:口袋蛋白随机失活一个
input_drop_rate = 0.5
select_pkt_rate = 0

batch_size = 32
n_epoch = 40
interrupt = None
save_best_epoch = 5  #  when `save_best_epoch` is reached and the loss starts to decrease, save best model parameters

# 大部分情况下，设置这个 flag 可以让内置的 cuDNN 的 auto-tuner 自动寻找最适合当前配置的高效算法，来达到优化运行效率的问题。
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

torch.manual_seed(seed)  # 让torch和numpy生成特定随机数，可复现结果
np.random.seed(seed)

In [3]:
f_param = open(path / 'parameters.txt', 'w')  # 用于记录超参数

print(f'device={device}')
print(f'seed={seed}')
print(f'write to {path}')
f_param.write(f'device={device}\n'
              f'seed={seed}\n'
              f'write to {path}\n')
               
print(f'max_seq_len={max_seq_len}\n'
      f'max_pkt_len={max_pkt_len}\n'
      f'max_smi_len={max_smi_len}\n')
f_param.write(f'max_seq_len={max_seq_len}\n'
              f'max_pkt_len={max_pkt_len}\n'
              f'max_smi_len={max_smi_len}\n')

print(f'repeat={repeat}\n'
      f'fix={fix}\n'
      f'onehot={onehot}\n'
      f'mode={mode}\n'
      f'input_drop_rate={input_drop_rate}\n'
      f'select_pkt_rate={select_pkt_rate}')
f_param.write(f'repeat={repeat}\n'
              f'onehot={onehot}\n'
              f'mode={mode}\n'
              f'input_drop_rate={input_drop_rate}\n'
              f'select_pkt_rate={select_pkt_rate}\n')

assert 0<save_best_epoch<n_epoch

model = DTIDenseNet3to1(onehot=onehot, dilation=dilation)
# model = DTIDenseNet3to2(onehot=onehot, dilation=dilation)
# model = DTIDenseNet2to1(onehot=onehot, dilation=dilation)
# model = DeepDTAF()
# model = DeepDTAF_3to2()

model = model.to(device)
model_p = sum([p.data.nelement() for p in model.parameters()])
print('weights:{}'.format(model_p))   # 计算参数量
f_param.write(f'model:{model_p} \n')
f_param.write(str(model)+'\n')
f_param.close()

data_loaders = {'training':
                    DataLoader(DTIDataset(data_path, f'fold_{fold}/training',
                                          max_seq_len, max_pkt_len, max_smi_len, 
                                          pkt_window=None, pkt_stride=None, 
                                          onehot=onehot, repeat=repeat, fix=fix,
                                          mode=mode, input_drop_rate=input_drop_rate, select_pkt_rate=select_pkt_rate),
                               batch_size=batch_size,
                               pin_memory=True,
                               num_workers=4,
                               shuffle=True),
                'validation':
                    DataLoader(DTIDataset(data_path, f'fold_{fold}/validation',
                                          max_seq_len, max_pkt_len, max_smi_len, 
                                          pkt_window=None, pkt_stride=None, 
                                          onehot=onehot),
                               batch_size=batch_size,
                               pin_memory=True,
                               num_workers=4,
                               shuffle=False),
                'test':
                    DataLoader(DTIDataset(data_path, f'fold_{fold}/test',
                                          max_seq_len, max_pkt_len, max_smi_len, 
                                          pkt_window=None, pkt_stride=None, 
                                          onehot=onehot),
                               batch_size=batch_size,
                               pin_memory=True,
                               num_workers=4,
                               shuffle=False)
               }

train_loader_no_repeat = DataLoader(DTIDataset(data_path, f'fold_{fold}/training', 
                                               max_seq_len, max_pkt_len, max_smi_len, 
                                               pkt_window=None, pkt_stride=None, 
                                               onehot=onehot),
                                    batch_size=batch_size,
                                    pin_memory=True,
                                    num_workers=4,
                                    shuffle=False)

# MultiStepLR, OneCycleLR
optimizer = optim.AdamW(model.parameters(), lr = 0.005)
scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.005, 
                                          epochs=n_epoch,
                                          steps_per_epoch=len(data_loaders['training']))   # 按照一种单周期函数设置学习率
# scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[10, 20, 40, 60, 75], gamma=0.5)   # 按照一种单周期函数设置学习率
        
tag = isinstance(model, DTIDenseNet3to2) or isinstance(model, DeepDTAF_3to2)
loss_function = Loss_fc(tag=tag)
print('special loss:', tag)

device=cuda:0
seed=18885
write to /data1/zxy/DTI Results/Dense433_3to1-drop_p_test/3to1_050000_r20_fold4_20220808105435_18885
max_seq_len=904
max_pkt_len=64
max_smi_len=154

repeat=20
fix=False
onehot=False
mode=3
input_drop_rate=0.5
select_pkt_rate=0
weights:165936
Dataset fold_4/training: will not fold pkt
Dataset fold_4/validation: will not fold pkt
Dataset fold_4/test: will not fold pkt
Dataset fold_4/training: will not fold pkt
special loss: False


In [ ]:
start = datetime.now()
print('start at ', start)
print('Train epoch: %d, Batch number per epoch: %d'%(n_epoch, len(data_loaders['training'])))

epoch_train_loss, epoch_val_loss = [], []
epoch_train_diff, epoch_val_diff = [], []
epoch_train_c_index, epoch_val_c_index = [], []
epoch_train_RMSE, epoch_val_RMSE = [], []
epoch_train_MAE, epoch_val_MAE = [], []
epoch_train_SD, epoch_val_SD = [], []
epoch_train_CORR, epoch_val_CORR = [], []
Lr = []

best_epoch = -1
best_val_loss = 100000000
for epoch in range(1, n_epoch + 1):
    
    batch_num = len(data_loaders['training'])
    for idx, (*x, y, name) in enumerate(data_loaders['training']):
        model.train()
        lr = optimizer.param_groups[0]['lr']
        Lr.append(lr)

        for i in range(len(x)):
            x[i] = x[i].to(device)
        y = y.to(device)

        optimizer.zero_grad()
        output = model(*x)
        loss = loss_function.calc_loss(output, y)
        if isinstance(model, DTIDenseNet3to2) or isinstance(model, DeepDTAF_3to2):
            loss[0].backward()
            if (idx+1)%100 == 0:
                print(f'Epoch {epoch}|{n_epoch}, Batch {idx+1}|{batch_num}, Loss:{loss[1].item()}, Diff:{loss[2].item()}')            
        else:
            loss.backward()
            if (idx+1)%100 == 0:
                print(f'Epoch {epoch}|{n_epoch}, Batch {idx+1}|{batch_num}, Loss:{loss.item()}')
                
        optimizer.step()
        scheduler.step()
        
    with torch.no_grad():
        for _p in ['training', 'validation']:

            if _p=='training':
                performance = test(model, train_loader_no_repeat, loss_function, device)  # 是个字典，包括{'loss','c_index'，等各种指标}
                epoch_train_loss.append(performance['loss'])
                epoch_train_diff.append(performance['diff'])
                epoch_train_c_index.append(performance['c_index'])
                epoch_train_RMSE.append(performance['RMSE'])
                epoch_train_MAE.append(performance['MAE'])
                epoch_train_SD.append(performance['SD'])
                epoch_train_CORR.append(performance['CORR'])

            else:
                performance = test(model, data_loaders[_p], loss_function, device)  # 是个字典，包括{'loss','c_index'，等各种指标}
                epoch_val_loss.append(performance['loss'])
                epoch_val_diff.append(performance['diff'])
                epoch_val_c_index.append(performance['c_index'])
                epoch_val_RMSE.append(performance['RMSE'])
                epoch_val_MAE.append(performance['MAE'])
                epoch_val_SD.append(performance['SD'])
                epoch_val_CORR.append(performance['CORR'])

            if _p=='validation' and epoch>=save_best_epoch and performance['loss']<best_val_loss:
                best_val_loss = performance['loss']
                best_epoch = epoch
                torch.save(model.state_dict(), path / 'best_model.pt')
    
#     scheduler.step()
    print(f'Epoch {epoch}|{n_epoch}: Lr -- {lr}, Train loss | diff -- {epoch_train_loss[-1]:.3f} | {epoch_train_diff[-1]:.3f}, Val loss | diff -- {epoch_val_loss[-1]:.3f} | {epoch_val_diff[-1]:.3f}') 

print('training finished')
            
model.load_state_dict(torch.load(path / 'best_model.pt'))
with open(path / 'result.txt', 'w') as f:
    f.write(f'best model found at epoch NO.{best_epoch}\n')
    for _p in ['training', 'validation', 'test',]:
        if _p == 'training':
            performance = test(model, train_loader_no_repeat, loss_function, device, path/(_p+'_predictions.xls'), save=True)
        else:
            performance = test(model, data_loaders[_p], loss_function, device, path/(_p+'_predictions.xls'), save=True)
        f.write(f'{_p}:\n')
        print(f'{_p}:')
        for k, v in performance.items():
            f.write(f'{k}: {v}\n')
            print(f'{k}: {v}')
        f.write('\n')
        print()

print('testing finished')
        
end = datetime.now()
print('end at:', end)
print('time used:', str(end - start))

start at  2022-08-08 10:54:44.634204
Train epoch: 40, Batch number per epoch: 9495
Epoch 1|40, Batch 100|9495, Loss:230.4127960205078
Epoch 1|40, Batch 200|9495, Loss:190.53834533691406
Epoch 1|40, Batch 300|9495, Loss:151.54603576660156
Epoch 1|40, Batch 400|9495, Loss:149.61138916015625
Epoch 1|40, Batch 500|9495, Loss:164.4099884033203
Epoch 1|40, Batch 600|9495, Loss:110.19918823242188
Epoch 1|40, Batch 700|9495, Loss:92.08783721923828
Epoch 1|40, Batch 800|9495, Loss:111.26994323730469


In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=[10,6])
plt.plot(epoch_train_loss)
plt.plot(epoch_val_loss)
plt.legend(['train loss', 'val loss'])
plt.show()

plt.figure(figsize=[10,6])
plt.plot(epoch_train_c_index)
plt.plot(epoch_val_c_index)
plt.legend(['train ci', 'val ci'])
plt.show()

plt.figure(figsize=[10,6])
plt.plot(epoch_train_diff)
plt.plot(epoch_val_diff)
plt.legend(['train diff', 'val diff'])
plt.show()